In [2]:
import pandas as pd
import glob
import os
import boto3
import json
import configparser
from botocore.exceptions import ClientError
import psycopg2
import time

In [4]:
def bucket_s3_exists(b):
    s3 = boto3.resource('s3')
    return s3.Bucket(b) in s3.buckets.all()

def create_s3_bucket(b, folders):
    client = boto3.client('s3')
    client.create_bucket(Bucket=b, CreateBucketConfiguration={'LocationConstraint': 'us-east-2'})
    if folders is not '':
        fls = folders.split(',')
        for f in fls:
            client.put_object(Bucket= b, Body='', Key=f + '/')

def upload_files_to_s3(file_name, b, folder, object_name, args=None):
    
    client = boto3.client('s3')

    if object_name is None:
        object_name = folder + "/{fname}".format(fname= os.path.basename(file_name)) 

    response = client.upload_file(file_name, b, object_name , ExtraArgs = args)

    return response


ACLargs = None
tt= time.time_ns()
prefix = f'uber-tracking-expenses-bucket-s3-{tt}'
bucket_names = {prefix: 'unprocessed_receipts,eats,rides'}

print('Creating the S3 buckets...')
for k in bucket_names:
    if not bucket_s3_exists(k):
       create_s3_bucket(k, bucket_names[k])    

print('S3 buckets created')

print('Uploading the local receipts files to uber-tracking-expenses-bucket-s3 AWS S3 bucket...')
files = glob.glob("E:/uber-expenses-tracking-main/data/receipts/*")

for file in files:
    print("Uploading file:", file)
    print(upload_files_to_s3(file, prefix, 'unprocessed_receipts', None, ACLargs))


print(f'Files uploaded to {prefix} AWS S3 bucket')
print(f'ID: {tt}')

<>:8: SyntaxWarning: "is not" with 'str' literal. Did you mean "!="?
C:\Users\vinay\AppData\Local\Temp\ipykernel_23028\2242547928.py:8: SyntaxWarning: "is not" with 'str' literal. Did you mean "!="?
  if folders is not '':


Creating the S3 buckets...
S3 buckets created
Uploading the local receipts files to uber-tracking-expenses-bucket-s3 AWS S3 bucket...
Uploading file: E:/uber-expenses-tracking-main/data/receipts\eats_0000001.eml
None
Uploading file: E:/uber-expenses-tracking-main/data/receipts\eats_0000002.eml
None
Uploading file: E:/uber-expenses-tracking-main/data/receipts\eats_0000003.eml
None
Uploading file: E:/uber-expenses-tracking-main/data/receipts\eats_0000004.eml
None
Uploading file: E:/uber-expenses-tracking-main/data/receipts\eats_0000005.eml
None
Uploading file: E:/uber-expenses-tracking-main/data/receipts\eats_0000006.eml
None
Uploading file: E:/uber-expenses-tracking-main/data/receipts\eats_0000007.eml
None
Uploading file: E:/uber-expenses-tracking-main/data/receipts\eats_0000008.eml
None
Uploading file: E:/uber-expenses-tracking-main/data/receipts\eats_0000009.eml
None
Uploading file: E:/uber-expenses-tracking-main/data/receipts\eats_0000010.eml
None
Uploading file: E:/uber-expenses-tra

# **Loading all the Params from the *dwh.cfg* file**

In [ ]:
from dotenv import load_dotenv
import os
import configparser
import pandas as pd

load_dotenv(override=True)

config = configparser.ConfigParser()

config["AWS"] = {
    "KEY": os.getenv("KEY"),
    "SECRET": os.getenv("SECRET")
}

config["DWH"] = {
    "DWH_CLUSTER_TYPE": os.getenv("DWH_CLUSTER_TYPE"),
    "DWH_NUM_NODES": os.getenv("DWH_NUM_NODES"),
    "DWH_NODE_TYPE": os.getenv("DWH_NODE_TYPE"),
    "DWH_IAM_ROLE_NAME": os.getenv("DWH_IAM_ROLE_NAME"),
    "DWH_CLUSTER_IDENTIFIER": os.getenv("DWH_CLUSTER_IDENTIFIER"),
    "DWH_DB": os.getenv("DWH_DB"),
    "DWH_DB_USER": os.getenv("DWH_DB_USER"),
    "DWH_DB_PASSWORD": os.getenv("DWH_DB_PASSWORD"),
    "DWH_PORT": os.getenv("DWH_PORT")
}

# Read values from the config object (same as the original notebook)
KEY = config.get("AWS", "KEY")
SECRET = config.get("AWS", "SECRET")

DWH_CLUSTER_TYPE = config.get("DWH", "DWH_CLUSTER_TYPE")
DWH_NUM_NODES = config.get("DWH", "DWH_NUM_NODES")
DWH_NODE_TYPE = config.get("DWH", "DWH_NODE_TYPE")
DWH_IAM_ROLE_NAME = config.get("DWH", "DWH_IAM_ROLE_NAME")
DWH_CLUSTER_IDENTIFIER = config.get("DWH", "DWH_CLUSTER_IDENTIFIER")
DWH_DB = config.get("DWH", "DWH_DB")
DWH_DB_USER = config.get("DWH", "DWH_DB_USER")
DWH_DB_PASSWORD = config.get("DWH", "DWH_DB_PASSWORD")
DWH_PORT = config.get("DWH", "DWH_PORT")

# Verify the values
pd.DataFrame({
    "Param": [
        "DWH_CLUSTER_TYPE",
        "DWH_NUM_NODES",
        "DWH_NODE_TYPE",
        "DWH_CLUSTER_IDENTIFIER",
        "DWH_DB",
        "DWH_DB_USER",
        "DWH_DB_PASSWORD",
        "DWH_PORT",
        "DWH_IAM_ROLE_NAME"
    ],
    "Value": [
        DWH_CLUSTER_TYPE,
        DWH_NUM_NODES,
        DWH_NODE_TYPE,
        DWH_CLUSTER_IDENTIFIER,
        DWH_DB,
        DWH_DB_USER,
        DWH_DB_PASSWORD,
        DWH_PORT,
        DWH_IAM_ROLE_NAME
    ]
})

# **Creating clients for *IAM*, *EC2* and *Redshift* ressources**

In [7]:
ec2 = boto3.resource('ec2',
                       region_name="us-east-2",
                       aws_access_key_id=KEY,
                       aws_secret_access_key=SECRET
                    )

iam = boto3.client('iam',aws_access_key_id=KEY,
                     aws_secret_access_key=SECRET,
                     region_name='us-east-2'
                  )

redshift = boto3.client('redshift',
                       region_name="us-east-2",
                       aws_access_key_id=KEY,
                       aws_secret_access_key=SECRET
                       )

# **Creating the *IAM* Role that makes *Redshift* able to access S3 buckets (ReadOnly)**

In [ ]:
try:
    print("Creating new IAM Role") 
    dwhRole = iam.create_role(
        Path='/',
        RoleName=DWH_IAM_ROLE_NAME,
        Description = "Allows Redshift clusters to call AWS services on your behalf.",
        AssumeRolePolicyDocument=json.dumps(
            {'Statement': [{'Action': 'sts:AssumeRole',
               'Effect': 'Allow',
               'Principal': {'Service': 'redshift.amazonaws.com'}}],
             'Version': '2012-10-17'})
    )    
except Exception as e:
    print(e)
    
    
print("Attaching Policy")

try:
    iam.attach_role_policy(RoleName=DWH_IAM_ROLE_NAME,
                       PolicyArn="arn:aws:iam::aws:policy/AmazonS3ReadOnlyAccess"
                      )['ResponseMetadata']['HTTPStatusCode']
    print("Get the IAM role ARN")
    roleArn = iam.get_role(RoleName=DWH_IAM_ROLE_NAME)['Role']['Arn']

    print(roleArn)                      
except Exception as e:
    print(e)                      


# **Creating *Redshift* Cluster**

In [14]:
try:
    response = redshift.create_cluster(        
        
        ClusterType=DWH_CLUSTER_TYPE,
        NodeType=DWH_NODE_TYPE,
        NumberOfNodes=int(DWH_NUM_NODES),


        DBName=DWH_DB,
        ClusterIdentifier=DWH_CLUSTER_IDENTIFIER,
        MasterUsername=DWH_DB_USER,
        MasterUserPassword=DWH_DB_PASSWORD,
        
   
        IamRoles=[roleArn]  
    )
except Exception as e:
    print(e)

#**Redshift Cluster Details** (Run ths piece of code several times until status show **Available**)

In [ ]:
def prettyRedshiftProps(props):
    pd.set_option('display.max_colwidth', None)
    keysToShow = ["ClusterIdentifier", "NodeType", "ClusterStatus", "MasterUsername", "DBName", "Endpoint", "NumberOfNodes", 'VpcId']
    x = [(k, v) for k,v in props.items() if k in keysToShow]
    return pd.DataFrame(data=x, columns=["Key", "Value"])

myClusterProps = redshift.describe_clusters(ClusterIdentifier=DWH_CLUSTER_IDENTIFIER)['Clusters'][0]
prettyRedshiftProps(myClusterProps)

# **Redshift Cluster endpoint and role ARN**

In [ ]:
DWH_ENDPOINT = myClusterProps['Endpoint']['Address']
DWH_ROLE_ARN = myClusterProps['IamRoles'][0]['IamRoleArn']
print("DWH_ENDPOINT :: ", DWH_ENDPOINT)
print("DWH_ROLE_ARN :: ", DWH_ROLE_ARN)

# **Incoming TCP port to access to the cluster endpoint**

In [18]:
try:
    vpc = ec2.Vpc(id=myClusterProps['VpcId'])
    defaultSg = list(vpc.security_groups.all())[0]
    print(defaultSg)
    defaultSg.authorize_ingress(
        GroupName=defaultSg.group_name,
        CidrIp='0.0.0.0/0',
        IpProtocol='TCP',
        FromPort=int(DWH_PORT),
        ToPort=int(DWH_PORT)
    )
except Exception as e:
    print(e)

ec2.SecurityGroup(id='sg-0b5b1d0c5e81c56f2')


#**Checking the connection to the redshift cluster**

In [ ]:
conn_string="postgresql://{}:{}@{}:{}/{}".format(DWH_DB_USER, DWH_DB_PASSWORD, DWH_ENDPOINT, DWH_PORT,DWH_DB)
print(conn_string)


In [ ]:
print('Connecting to RedShift', conn_string)
conn = psycopg2.connect(conn_string)
print('Connected to Redshift')

# **Cleaning and deleting all the resources** (Do not run the next lines until finish your experiments)

In [ ]:
# #### CAREFUL!!
# #-- Uncomment & run to delete the created resources
# redshift.delete_cluster( ClusterIdentifier=DWH_CLUSTER_IDENTIFIER,  SkipFinalClusterSnapshot=True)
# #### CAREFUL!!

In [ ]:
# myClusterProps = redshift.describe_clusters(ClusterIdentifier=DWH_CLUSTER_IDENTIFIER)['Clusters'][0]
# prettyRedshiftProps(myClusterProps)

In [ ]:
# iam.detach_role_policy(RoleName=DWH_IAM_ROLE_NAME, PolicyArn="arn:aws:iam::aws:policy/AmazonS3ReadOnlyAccess")
# iam.delete_role(RoleName=DWH_IAM_ROLE_NAME)